In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import lal

project_path = '~/projects/npe'

npe_dir = os.path.abspath(os.path.expanduser(os.path.join(project_path, 'npe')))
if npe_dir not in sys.path:
    sys.path.append(npe_dir)

from waveform_analysis import VAE

MSUN_KM = lal.MSUN_SI * lal.G_SI / lal.C_SI ** 2 / 1e3
MSUN_S  = MSUN_KM / lal.C_SI * 1e3

/tmp/ipykernel_4102738/2656921303.py:6: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
from workflow import initialize

# Import PyTorch network
def import_network(network_path, base_path=project_path):

    network_fullpath = os.path.expanduser(os.path.join(base_path, network_path))

    device = torch.device('cpu')
    model_kwargs = dict(depth=4, width=512, data_dim=640, grid_dim=2, freeze_shape=True)
    optimizer_kwargs = dict(lr=1e-4, weight_decay=1e-4)
    scheduler_kwargs = dict(gamma=0.9)
    
    model_type = VAE
    optimizer_type = torch.optim.AdamW
    scheduler_type = torch.optim.lr_scheduler.ExponentialLR

    model, optimizer, scheduler = initialize(
                                    model_type, optimizer_type, scheduler_type,
                                    model_kwargs=model_kwargs,
                                    optimizer_kwargs=optimizer_kwargs,
                                    scheduler_kwargs=scheduler_kwargs, device=device)
    
    state_dict = torch.load(network_fullpath, map_location=device, weights_only=True)

    model = model_type(**model_kwargs).to(device)
    optimizer = optimizer_type(model.parameters(), **optimizer_kwargs)
    scheduler = scheduler_type(optimizer, **scheduler_kwargs)

    model.load_state_dict(state_dict['model'])
    if optimizer is not None: optimizer.load_state_dict(state_dict['optimizer'])
    if scheduler is not None: scheduler.load_state_dict(state_dict['scheduler'])

    model.eval()
    model.train(False)

    return model, device, optimizer, scheduler

In [3]:
def save_state(path, model, optimizer, scheduler):
    import torch
    state_dict = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
    }
    torch.save(state_dict, path)

In [4]:
# Example network paths
example_network_path = './npe/utils/param_estimation_test/npe_network.pt'   # Example network
BBH_network_path = './network/checkpoints/npe-BH/npe-BH_100-epochs.pt'  # BBH network
BBH_network_path_3 = './network/checkpoints/npe-BH-3/npe-BH_100-epochs.pt'  # BBH network (bad seed)
BBH_network_path_4 = './network/checkpoints/npe-BH-4/npe-BH_100-epochs.pt'  # BBH network (good seed)
BBH_network_path_4_2 = './network/checkpoints/npe-BH-4-2/npe-BH_100-epochs.pt'  # BBH network (good seed ?)
BNS_network_path = './network/checkpoints/npe-NS/npe-NS_100-epochs.pt'  # BNS network
BNS_network_path_1 = './network/checkpoints/npe-NS-1/npe-NS_100-epochs.pt'  # BNS network

In [5]:
def resave_network(network_path_in, base_path=project_path):
    nwk = import_network(network_path_in)
    network_fullpath = os.path.expanduser(os.path.join(base_path, network_path_in)).replace('.pt','_NEW.pt')
    print(network_fullpath)
    save_state(network_fullpath, nwk[0], nwk[2], nwk[3])

In [9]:
BBH_network_path_9 = './network/checkpoints/npe-BH-9/npe-BH_100-epochs.pt'  # BBH network
resave_network(BBH_network_path_9)

/u/loane2/projects/npe/./network/checkpoints/npe-BH-9/npe-BH_100-epochs_NEW.pt


In [8]:
BNS_network_path_4 = './network/checkpoints/npe-NS-4/npe-NS_100-epochs.pt'  # BNS network
resave_network(BNS_network_path_4)

/u/loane2/projects/npe/./network/checkpoints/npe-NS-4/npe-NS_100-epochs_NEW.pt
